In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
%matplotlib inline

In [ ]:
import os
# Reads a local copy if you have one, otherwise downloads the dataset from GitHub
DATA_URL = "https://raw.githubusercontent.com/krishnaik06/playstore-Dataset/main/googleplaystore.csv"
data = pd.read_csv("googleplaystore.csv" if os.path.exists("googleplaystore.csv") else DATA_URL)
data.head()

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
data.isna().sum()

In [ ]:
median_rating = data['Rating'].median()
data['Rating'] = data['Rating'].fillna(median_rating)
print("Rating NaNs Imputed. Non-null count:", data['Rating'].count())

In [ ]:
data["Reviews"].unique()

In [ ]:
data["Reviews"].str.isnumeric().sum()

In [ ]:
data[~data["Reviews"].str.isnumeric()]

In [ ]:
# Drop the one corrupted row (its columns are shifted). Label-based and safe to re-run.
data = data.drop(index=10472, errors="ignore")

In [ ]:
data["Reviews"]=data["Reviews"].astype(int)

In [ ]:
data.info()

In [ ]:
data["Size"].unique()

In [ ]:
data["Size"].isnull().sum()

In [ ]:
def size_to_mb(s):
    """Convert strings like '19M' or '201k' to megabytes. 'Varies with device' -> NaN."""
    if pd.isna(s) or s == 'Varies with device':
        return np.nan
    s = s.strip()
    if s.endswith('M'):
        return float(s[:-1].replace(',', ''))
    if s.endswith(('k', 'K')):
        return float(s[:-1].replace(',', '')) / 1024
    return np.nan

data['Size_MB'] = data['Size'].apply(size_to_mb)
print("Verifying the first 5 entries of the new column:")
print(data[['Size', 'Size_MB']].head())

In [ ]:
data["Size_MB"]

In [ ]:
median_size = data['Size_MB'].median()
data['Size_MB'] = data['Size_MB'].fillna(median_size)

In [ ]:
data["Size_MB"]

In [ ]:
data["Installs"].unique()

In [ ]:
data["Price"].unique()

In [ ]:
chars_to_remove=['+',',','$']
cols_to_clean=['Installs','Price']
for item in chars_to_remove:
    for cols in cols_to_clean:
        data[cols]=data[cols].str.replace(item,'',regex=False)

In [ ]:
data["Price"].unique()

In [ ]:
data["Installs"].unique()

In [ ]:
data["Price"]=data["Price"].astype(float)
data["Installs"]=data["Installs"].astype(int)

In [ ]:
data.info()

In [ ]:
data["Last Updated"].unique()

In [ ]:
data["Last Updated"]=pd.to_datetime(data["Last Updated"])
data["Day"]=data["Last Updated"].dt.day
data["Month"]=data["Last Updated"].dt.month
data["Year"]=data["Last Updated"].dt.year

In [ ]:
data.info()

In [ ]:
data.head()

In [ ]:
data.drop(columns="Last Updated",inplace=True)

In [ ]:
data.head()

In [ ]:
data["Current Ver"].unique()

In [ ]:
data['Current Ver'] = data['Current Ver'].replace('Varies with device', np.nan)
data['Current Ver'] = data['Current Ver'].str.extract(r'^(\d+)', expand=False)
data['Current Ver'] = pd.to_numeric(data['Current Ver'], errors='coerce').fillna(1.0)

In [ ]:
data["Current Ver"].unique()

In [ ]:
data["Android Ver"].unique()

In [ ]:
data['Android Ver'] = data['Android Ver'].replace('Varies with device', np.nan)
data['Android Ver'] = data['Android Ver'].str.replace(' and up', '', regex=False).str.split('.').str[0]
data['Android Ver'] = pd.to_numeric(data['Android Ver'], errors='coerce').fillna(4.0)

In [ ]:
data["Android Ver"].unique()

In [ ]:
data.info()

In [ ]:
data[data.duplicated('App')]

In [ ]:
data[data.duplicated('App')].shape

In [ ]:
data=data.drop_duplicates(subset=['App'],keep='first').copy()

In [ ]:
data.shape

In [ ]:
# One-hot encode Category. The rest of the notebook expects columns named Category_Cleaned_<name>.
category_dummies = pd.get_dummies(data['Category'], prefix='Category_Cleaned', dtype=int)
data = pd.concat([data, category_dummies], axis=1)
print("Category columns created:", category_dummies.shape[1])

In [ ]:
paid_data = data[data['Type'] == 'Paid'].copy()

In [ ]:
data['Installs_log'] = np.log1p(data['Installs'])
data['Reviews_log'] = np.log1p(data['Reviews'])

print("Log transformation columns created successfully.")
print(data[['Installs_log', 'Reviews_log']].head())

In [ ]:
# Check if 'Installs' is an object (string) before running string replacement
if data['Installs'].dtype == 'object':
    data['Installs'] = data['Installs'].str.replace('+', '', regex=False).str.replace(',', '', regex=False)
    data = data[data['Installs'] != 'Free']
    data['Installs'] = pd.to_numeric(data['Installs'])
else:
    print("'Installs' column already cleaned (numeric). Skipping string operations.")

# Ensure 'Reviews' is numeric (coercing errors, filling NaNs is not needed for log transformation if NaNs are present)
data['Reviews'] = pd.to_numeric(data['Reviews'], errors='coerce')

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(data['Installs_log'], kde=True)
plt.title('Distribution of Log(Installs)')
plt.xlabel(r'$\log(1 + \text{Installs})$')

plt.subplot(1, 2, 2)
sns.scatterplot(x='Reviews_log', y='Installs_log', data=data, alpha=0.5)
plt.title('Log(Reviews) vs. Log(Installs)')
plt.xlabel(r'$\log(1 + \text{Reviews})$')
plt.ylabel(r'$\log(1 + \text{Installs})$')
plt.tight_layout()
plt.savefig('installs_log_distributions.png')
plt.show()

print("\nSuccessfully generated 'installs_log_distributions.png'")

In [ ]:
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
sns.histplot(data['Installs'], bins=50, kde=True, log_scale=True)
plt.title('Distribution of App Installations (Log Scale)')
plt.xlabel('Installs (Log Scale)')

plt.subplot(1, 2, 2)
sns.histplot(data[data['Price'] > 0]['Price'], bins=30, kde=True)
plt.title('Distribution of App Price (Non-Zero)')
plt.xlabel('Price ($)')

plt.tight_layout()

In [ ]:
category_cols = [col for col in data.columns if col.startswith('Category_Cleaned_')]
category_median_price = {}

for col in category_cols:
    median_price = paid_data[paid_data[col] == 1]['Price'].median()
    category_median_price[col.replace('Category_Cleaned_', '')] = median_price

price_df = pd.DataFrame(category_median_price.items(), columns=['Category', 'Median_Price']).sort_values(by='Median_Price', ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x='Median_Price', y='Category', data=price_df, palette='viridis')
plt.title('Top 10 Categories by Median App Price (Paid Apps)')
plt.xlabel('Median Price (USD)')
plt.ylabel('Category')
plt.savefig('median_price_by_category.png')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='Content Rating', y='Installs_log', data=data, order=data['Content Rating'].value_counts().index)
plt.xticks(rotation=45, ha='right')
plt.title('Log(Installs) by Content Rating')
plt.xlabel('Content Rating')
plt.ylabel(r'$\log(1 + \text{Installs})$')
plt.tight_layout()
plt.savefig('installs_by_content_rating.png')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
# We use paid_data here and bin the Ratings to make the visualization cleaner
paid_data['Rating_Group'] = pd.cut(paid_data['Rating'], bins=[0, 3.5, 4.0, 4.5, 5.0], labels=['<3.5', '3.5-4.0', '4.0-4.5', '4.5-5.0'], right=True)
sns.boxplot(x='Rating_Group', y='Price', data=paid_data)
plt.title('Price Distribution by Rating Group (Paid Apps)')
plt.xlabel('Rating Group')
plt.ylabel('Price (USD)')
plt.savefig('price_by_rating_group.png')
plt.show()

In [ ]:
plt.figure(figsize=(15, 5))
plt.subplot(1, 2, 1)

sns.scatterplot(x=np.log1p(data['Reviews']), y=np.log1p(data['Installs']), alpha=0.6)
plt.title('Log(Reviews) vs Log(Installs)')
plt.xlabel('Log(Reviews)')
plt.ylabel('Log(Installs)')

plt.subplot(1, 2, 2)
sns.scatterplot(x=data['Rating'], y=np.log1p(data['Installs']), alpha=0.6)
plt.title('Rating vs Log(Installs)')
plt.xlabel('Rating')
plt.ylabel('Log(Installs)')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))
top_cats = data.groupby('Category')['Installs_log'].mean().sort_values(ascending=False).head(15)
sns.barplot(x=top_cats.index, y=top_cats.values, hue=top_cats.index, palette="viridis", legend=False)
plt.xticks(rotation=70, ha='right')
plt.title('Average Log(Installs) by Top 15 Categories')
plt.xlabel('Category')
plt.ylabel('Average Log(Installs)')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))
sns.violinplot(x='Type', y='Rating', data=data, hue='Type',
               palette={'Free': 'skyblue', 'Paid': 'salmon'}, legend=False)
plt.title('Rating Distribution by App Type')
plt.xlabel('Type')
plt.ylabel('Rating')
plt.show()

In [ ]:
numerical_features = ['Rating', 'Reviews', 'Installs', 'Price']
# Note: 'Current Ver' and 'Android Ver' are not useful for correlation and should be excluded.
features_to_correlate = numerical_features + ['Type']

correlation_data = data[features_to_correlate].copy()

correlation_data['Type_Encoded'] = correlation_data['Type'].apply(lambda x: 1 if x == 'Paid' else 0)
correlation_data = correlation_data.drop(columns=['Type'])

for col in ['Reviews', 'Installs']:
    correlation_data[col] = np.log1p(correlation_data[col])


plt.figure(figsize=(10, 8))
sns.heatmap(correlation_data.corr(), annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Feature Correlation Matrix (Log-Transformed Installs & Reviews)')
plt.savefig('correlation_matrix.png')
plt.show()

print("The correlation matrix plot has been successfully generated as 'correlation_matrix.png'.")

In [ ]:
print(data.info())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import joblib # Required for saving models for Streamlit

X_installs_features = ['Reviews_log', 'Size_MB', 'Rating', 'Price']
category_cols = [col for col in data.columns if col.startswith('Category_Cleaned_')]
X_installs_features.extend(category_cols)
X_installs_features.extend(['Type', 'Content Rating'])

X_installs_temp = data[X_installs_features].copy()
X_installs = pd.get_dummies(X_installs_temp, columns=['Type', 'Content Rating'], drop_first=True)
y_installs = data['Installs_log']
X_installs = X_installs.fillna(X_installs.median())
installs_features = X_installs.columns.tolist()

X_train_installs, X_test_installs, y_train_installs, y_test_installs = train_test_split(
    X_installs, y_installs, test_size=0.25, random_state=42
)

lr_model = LinearRegression()
lr_model.fit(X_train_installs, y_train_installs)

In [ ]:
y_pred_installs = lr_model.predict(X_test_installs)
r2_installs = r2_score(y_test_installs, y_pred_installs)
print(f"Installs Model R2: {r2_installs:.4f}")

In [ ]:
# ---- Price model (paid apps only) ----
# Prices above $50 are extreme outliers (e.g. the $399.99 novelty apps). They dominate a linear
# model, so they are excluded. We also predict log(price) and convert back with expm1.
paid_data = data[(data['Type'] == 'Paid') & (data['Price'] <= 50)].copy()
X_price_features = ['Reviews_log', 'Rating', 'Size_MB']
category_cols_price = [col for col in paid_data.columns if col.startswith('Category_Cleaned_')]
X_price_features.extend(category_cols_price)

X_price = paid_data[X_price_features].copy()
y_price = np.log1p(paid_data['Price'])
X_price = X_price.fillna(X_price.median())
price_features = X_price.columns.tolist()

X_train_price, X_test_price, y_train_price, y_test_price = train_test_split(
    X_price, y_price, test_size=0.25, random_state=42
)

lr_price_model = LinearRegression()
lr_price_model.fit(X_train_price, y_train_price)
print("Paid apps used for the price model:", len(paid_data))

In [ ]:
y_pred_price = np.expm1(lr_price_model.predict(X_test_price))
y_true_price = np.expm1(y_test_price)

mae_price = mean_absolute_error(y_true_price, y_pred_price)
baseline_price = np.full(len(y_true_price), np.median(np.expm1(y_train_price)))
baseline_mae = mean_absolute_error(y_true_price, baseline_price)

print(f"Price Model MAE: ${mae_price:.2f}")
print(f"Baseline MAE (always predict the median price): ${baseline_mae:.2f}")
print(f"Price Model R2: {r2_score(y_true_price, y_pred_price):.4f}")

In [ ]:
joblib.dump(lr_model, 'installs_model.pkl')
joblib.dump(installs_features, 'installs_features.pkl')

joblib.dump(lr_price_model, 'price_model.pkl')
joblib.dump(price_features, 'price_features.pkl')

import json
metrics = {
    "installs_r2": float(r2_installs),
    "price_mae": float(mae_price),
    "price_baseline_mae": float(baseline_mae),
    "price_r2": float(r2_score(y_true_price, y_pred_price)),
    "n_apps": int(len(data)),
    "n_paid_apps_used": int(len(paid_data)),
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("\nNotebook Complete! Models and feature lists are saved and ready for Streamlit deployment.")